
# Brownian Bridge Reference And Mixture Visualizations

This notebook builds presentation-oriented toy visualizations for three laws on a single interval $[t_{i-1}, t_i]$:

- the anchored reference / prior law with only the left endpoint fixed,
- the pinned Brownian bridge with both endpoints fixed,
- the local bridge mixture obtained by mixing endpoint-conditioned bridges over a terminal law.

It exports slide-ready PNG/PDF figures under `notebooks/figures/brownian_bridge_reference_mixture_viz/`.


In [15]:

from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.figure import Figure


def _detect_repo_root() -> Path:
    candidates: list[Path] = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    if "__file__" in globals():
        file_path = Path(__file__).resolve()
        candidates.extend([file_path.parent, *file_path.parent.parents])
    seen: set[Path] = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "mmsfm").exists() and (candidate / "csp").exists():
            return candidate
    raise RuntimeError("Could not locate repo root containing both 'mmsfm' and 'csp'.")


REPO_ROOT = _detect_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from csp._conditional_bridge import sample_brownian_bridge_state
from scripts.images.field_visualization import format_for_paper, publication_grid_figure_size

format_for_paper()

print(f"REPO_ROOT: {REPO_ROOT}")


REPO_ROOT: /data1/jy384/research/MMSFM



## Laws On One Interval

For a fixed left endpoint $z_{i-1}$ on $[t_{i-1}, t_i]$:

- Anchored reference / prior law:
  $$
  \mathbb Q_i^{z_{i-1}}(d\omega_i)
  =
  \mathbb Q(d\omega_i \mid Z_{t_{i-1}} = z_{i-1}).
  $$
- Fixed-endpoint bridge to one terminal state $y$:
  $$
  \mathbb Q(d\omega_i \mid Z_{t_{i-1}} = z_{i-1}, Z_{t_i} = y).
  $$
- Local bridge mixture over a terminal law $\gamma_i(\cdot \mid \zeta_{i-1})$:
  $$
  \mathbb P_i^\gamma(d\omega_i \mid \zeta_{i-1})
  =
  \int \mathbb Q(d\omega_i \mid Z_{t_{i-1}}=z_{i-1}, Z_{t_i}=y)\, \gamma_i(dy \mid \zeta_{i-1}).
  $$

The figures below use exact discrete-time Brownian motion and exact Brownian-bridge path construction on a fixed time grid. The bridge-mixture figures sample terminal endpoints first, then sample pinned bridges conditionally on those endpoints.



## Configuration

All settings can be overridden via environment variables before execution.

- `EXPORT_DIR`, `RNG_SEED`, `SIGMA`, `T_START`, `T_END`, `N_TIME`
- `ANCHORED_PATHS`, `PINNED_PATHS`, `MIXTURE_PATHS`
- `ONE_D_START`, `ONE_D_END`, `TWO_D_START`, `TWO_D_END`
- `DISCRETE_WEIGHTS`, `DISCRETE_ENDPOINTS_1D`, `DISCRETE_ENDPOINTS_2D`
- `GMM_WEIGHTS`, `GMM_MEANS_1D`, `GMM_STDS_1D`, `GMM_MEANS_2D`, `GMM_STDS_2D`
- `TRANSPARENT_EXPORT`, `FIG_DPI`


In [16]:

def _load_json_env(name: str, default: Any) -> Any:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return json.loads(raw)


EXPORT_DIR = Path(
    os.environ.get(
        "EXPORT_DIR",
        str(REPO_ROOT / "notebooks" / "figures" / "brownian_bridge_reference_mixture_viz"),
    )
).expanduser().resolve()
RNG_SEED = int(os.environ.get("RNG_SEED", "7"))
SIGMA = float(os.environ.get("SIGMA", "0.9"))
T_START = float(os.environ.get("T_START", "0.0"))
T_END = float(os.environ.get("T_END", "1.0"))
N_TIME = int(os.environ.get("N_TIME", "121"))
ANCHORED_PATHS = int(os.environ.get("ANCHORED_PATHS", "320"))
PINNED_PATHS = int(os.environ.get("PINNED_PATHS", "240"))
MIXTURE_PATHS = int(os.environ.get("MIXTURE_PATHS", "240"))
VALIDATION_SAMPLES = int(os.environ.get("VALIDATION_SAMPLES", "4096"))
TRANSPARENT_EXPORT = os.environ.get("TRANSPARENT_EXPORT", "1") != "0"
FIG_DPI = int(os.environ.get("FIG_DPI", "300"))

ONE_D_START = float(os.environ.get("ONE_D_START", "-1.25"))
ONE_D_END = float(os.environ.get("ONE_D_END", "1.5"))
TWO_D_START = np.asarray(_load_json_env("TWO_D_START", [-1.6, -0.4]), dtype=np.float64)
TWO_D_END = np.asarray(_load_json_env("TWO_D_END", [1.55, 0.85]), dtype=np.float64)

DISCRETE_WEIGHTS = np.asarray(_load_json_env("DISCRETE_WEIGHTS", [0.45, 0.35, 0.20]), dtype=np.float64)
DISCRETE_WEIGHTS = DISCRETE_WEIGHTS / DISCRETE_WEIGHTS.sum()
DISCRETE_ENDPOINTS_1D = np.asarray(_load_json_env("DISCRETE_ENDPOINTS_1D", [-1.15, 0.15, 1.55]), dtype=np.float64)
DISCRETE_ENDPOINTS_2D = np.asarray(
    _load_json_env("DISCRETE_ENDPOINTS_2D", [[0.85, 1.4], [1.95, 0.25], [1.0, -1.15]]),
    dtype=np.float64,
)

GMM_WEIGHTS = np.asarray(_load_json_env("GMM_WEIGHTS", [0.40, 0.35, 0.25]), dtype=np.float64)
GMM_WEIGHTS = GMM_WEIGHTS / GMM_WEIGHTS.sum()
GMM_MEANS_1D = np.asarray(_load_json_env("GMM_MEANS_1D", [-1.25, 0.55, 1.7]), dtype=np.float64)
GMM_STDS_1D = np.asarray(_load_json_env("GMM_STDS_1D", [0.22, 0.28, 0.18]), dtype=np.float64)
GMM_MEANS_2D = np.asarray(
    _load_json_env("GMM_MEANS_2D", [[0.9, 1.55], [2.05, 0.2], [0.95, -1.2]]),
    dtype=np.float64,
)
GMM_STDS_2D = np.asarray(_load_json_env("GMM_STDS_2D", [0.22, 0.28, 0.20]), dtype=np.float64)

MIXTURE_COLORS = ["#c44e52", "#dd8452", "#4c72b0"]
REFERENCE_COLOR = "#8c8c8c"
BRIDGE_COLOR = "#2b6cb0"


LOCAL_LAW_ONE_D_FIGSIZE = publication_grid_figure_size(
    1,
    2,
    column_span=1,
    width_fraction=0.98,
    panel_height_in=1.55,
    extra_height_in=0.18,
)
LOCAL_LAW_TWO_D_FIGSIZE = publication_grid_figure_size(
    1,
    2,
    column_span=1,
    width_fraction=0.98,
    panel_height_in=2.05,
    extra_height_in=0.20,
)

print(f"EXPORT_DIR          : {EXPORT_DIR}")
print(f"RNG_SEED            : {RNG_SEED}")
print(f"SIGMA               : {SIGMA}")
print(f"time interval       : [{T_START}, {T_END}]")
print(f"N_TIME              : {N_TIME}")
print(f"ANCHORED_PATHS      : {ANCHORED_PATHS}")
print(f"PINNED_PATHS        : {PINNED_PATHS}")
print(f"MIXTURE_PATHS       : {MIXTURE_PATHS}")
print(f"TRANSPARENT_EXPORT  : {TRANSPARENT_EXPORT}")


EXPORT_DIR          : /data1/jy384/research/MMSFM/notebooks/figures/brownian_bridge_reference_mixture_viz
RNG_SEED            : 7
SIGMA               : 0.9
time interval       : [0.0, 1.0]
N_TIME              : 121
ANCHORED_PATHS      : 320
PINNED_PATHS        : 240
MIXTURE_PATHS       : 240
TRANSPARENT_EXPORT  : True


In [17]:

def _save_figure(fig: Figure, base_path: Path, *, transparent: bool = TRANSPARENT_EXPORT) -> None:
    base_path.parent.mkdir(parents=True, exist_ok=True)
    fig.patch.set_alpha(0.0 if transparent else 1.0)
    fig.savefig(base_path.with_suffix(".png"), dpi=FIG_DPI, bbox_inches="tight", transparent=transparent)
    fig.savefig(base_path.with_suffix(".pdf"), bbox_inches="tight", transparent=transparent)
    plt.close(fig)


def _as_vector(x: float | list[float] | np.ndarray) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64)
    if arr.ndim == 0:
        arr = arr.reshape(1)
    return arr


def _time_grid() -> np.ndarray:
    if N_TIME < 3:
        raise ValueError("N_TIME must be at least 3.")
    if T_END <= T_START:
        raise ValueError("Require T_END > T_START.")
    return np.linspace(T_START, T_END, N_TIME, dtype=np.float64)


def _sample_brownian_motion_paths(
    time_grid: np.ndarray,
    x0: float | list[float] | np.ndarray,
    *,
    sigma: float,
    n_paths: int,
    rng: np.random.Generator,
) -> np.ndarray:
    x0_arr = _as_vector(x0)
    dt = np.diff(time_grid)
    noise = rng.normal(
        loc=0.0,
        scale=np.sqrt(dt)[None, :, None],
        size=(int(n_paths), dt.shape[0], x0_arr.shape[0]),
    )
    w = np.concatenate(
        [np.zeros((int(n_paths), 1, x0_arr.shape[0]), dtype=np.float64), np.cumsum(noise, axis=1)],
        axis=1,
    )
    return x0_arr[None, None, :] + float(sigma) * w


def _sample_brownian_bridge_paths(
    time_grid: np.ndarray,
    x_start: float | list[float] | np.ndarray,
    x_end: float | list[float] | np.ndarray,
    *,
    sigma: float,
    n_paths: int,
    rng: np.random.Generator,
) -> np.ndarray:
    x_start_arr = _as_vector(x_start)
    x_end_arr = np.asarray(x_end, dtype=np.float64)
    if x_end_arr.ndim == 0:
        x_end_arr = x_end_arr.reshape(1)
    if x_end_arr.ndim == 1:
        x_end_arr = np.broadcast_to(x_end_arr, (int(n_paths), x_start_arr.shape[0]))
    if x_end_arr.shape != (int(n_paths), x_start_arr.shape[0]):
        raise ValueError(
            f"Expected x_end with shape {(int(n_paths), x_start_arr.shape[0])}, got {x_end_arr.shape}."
        )
    tau = ((time_grid - time_grid[0]) / (time_grid[-1] - time_grid[0])).reshape(1, -1, 1)
    dt = np.diff(time_grid)
    noise = rng.normal(
        loc=0.0,
        scale=np.sqrt(dt)[None, :, None],
        size=(int(n_paths), dt.shape[0], x_start_arr.shape[0]),
    )
    w = np.concatenate(
        [np.zeros((int(n_paths), 1, x_start_arr.shape[0]), dtype=np.float64), np.cumsum(noise, axis=1)],
        axis=1,
    )
    bridge_noise = w - tau * w[:, -1:, :]
    deterministic = (1.0 - tau) * x_start_arr.reshape(1, 1, -1) + tau * x_end_arr[:, None, :]
    return deterministic + float(sigma) * bridge_noise


def _sample_discrete_endpoint_indices(weights: np.ndarray, *, n_samples: int, rng: np.random.Generator) -> np.ndarray:
    probs = np.asarray(weights, dtype=np.float64)
    probs = probs / probs.sum()
    return rng.choice(probs.shape[0], size=int(n_samples), p=probs)


def _sample_discrete_endpoints(
    endpoints: np.ndarray,
    weights: np.ndarray,
    *,
    n_samples: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    endpoint_arr = np.asarray(endpoints, dtype=np.float64)
    if endpoint_arr.ndim == 1:
        endpoint_arr = endpoint_arr[:, None]
    component_idx = _sample_discrete_endpoint_indices(weights, n_samples=n_samples, rng=rng)
    return endpoint_arr[component_idx], component_idx


def _sample_gmm_endpoints(
    means: np.ndarray,
    scales: np.ndarray,
    weights: np.ndarray,
    *,
    n_samples: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    mean_arr = np.asarray(means, dtype=np.float64)
    if mean_arr.ndim == 1:
        mean_arr = mean_arr[:, None]
    scale_arr = np.asarray(scales, dtype=np.float64)
    component_idx = _sample_discrete_endpoint_indices(weights, n_samples=n_samples, rng=rng)
    dim = mean_arr.shape[1]
    eps = rng.normal(size=(int(n_samples), dim))
    sampled = mean_arr[component_idx] + scale_arr[component_idx, None] * eps
    return sampled, component_idx


def _normal_pdf_1d(x: np.ndarray, mean: float, std: float) -> np.ndarray:
    z = (x - mean) / std
    return np.exp(-0.5 * z * z) / (std * np.sqrt(2.0 * np.pi))


def _mixture_pdf_1d(x: np.ndarray, means: np.ndarray, stds: np.ndarray, weights: np.ndarray) -> np.ndarray:
    density = np.zeros_like(x, dtype=np.float64)
    for w, m, s in zip(weights, means, stds, strict=True):
        density = density + float(w) * _normal_pdf_1d(x, float(m), float(s))
    return density


def _bridge_midpoint_moments(
    x_start: float,
    x_end: float,
    t: float,
    *,
    t_start: float,
    t_end: float,
    sigma: float,
) -> tuple[float, float]:
    alpha = (t_end - t) / (t_end - t_start)
    beta = (t - t_start) / (t_end - t_start)
    mean = alpha * x_start + beta * x_end
    var = sigma * sigma * (t - t_start) * (t_end - t) / (t_end - t_start)
    return float(mean), float(var)


def _log_brownian_transition_density(
    y: np.ndarray,
    x: np.ndarray,
    *,
    delta_t: float,
    sigma: float,
) -> np.ndarray:
    y_arr = np.asarray(y, dtype=np.float64)
    x_arr = np.asarray(x, dtype=np.float64)
    if y_arr.ndim == 1:
        y_arr = y_arr[None, :]
    scale2 = float(sigma) * float(sigma) * float(delta_t)
    dim = y_arr.shape[-1]
    diff = y_arr - x_arr.reshape(1, -1)
    return -0.5 * dim * np.log(2.0 * np.pi * scale2) - 0.5 * np.sum(diff * diff, axis=-1) / scale2


def _normalize_log_weights(log_weights: np.ndarray) -> np.ndarray:
    values = np.asarray(log_weights, dtype=np.float64)
    shifted = values - np.max(values)
    weights = np.exp(shifted)
    return weights / weights.sum()


def _component_representative_indices(component_idx: np.ndarray, per_component: int) -> np.ndarray:
    comp_arr = np.asarray(component_idx, dtype=np.int64)
    selected: list[int] = []
    for comp in range(len(MIXTURE_COLORS)):
        selected.extend(np.flatnonzero(comp_arr == comp)[: int(per_component)].tolist())
    return np.asarray(selected, dtype=np.int64)


def _component_color(idx: int) -> str:
    return MIXTURE_COLORS[int(idx) % len(MIXTURE_COLORS)]


In [18]:

time_grid = _time_grid()
mid_index = int(len(time_grid) // 2)
t_mid = float(time_grid[mid_index])
seed_seq = np.random.SeedSequence(RNG_SEED)
subseeds = seed_seq.spawn(20)
rngs = [np.random.default_rng(s) for s in subseeds]

anchored_1d = _sample_brownian_motion_paths(time_grid, ONE_D_START, sigma=SIGMA, n_paths=ANCHORED_PATHS, rng=rngs[0])
pinned_1d = _sample_brownian_bridge_paths(time_grid, ONE_D_START, ONE_D_END, sigma=SIGMA, n_paths=PINNED_PATHS, rng=rngs[1])
anchored_2d = _sample_brownian_motion_paths(time_grid, TWO_D_START, sigma=SIGMA, n_paths=ANCHORED_PATHS, rng=rngs[2])
pinned_2d = _sample_brownian_bridge_paths(time_grid, TWO_D_START, TWO_D_END, sigma=SIGMA, n_paths=PINNED_PATHS, rng=rngs[3])

discrete_end_1d, discrete_idx_1d = _sample_discrete_endpoints(DISCRETE_ENDPOINTS_1D, DISCRETE_WEIGHTS, n_samples=MIXTURE_PATHS, rng=rngs[4])
discrete_mix_1d = _sample_brownian_bridge_paths(time_grid, ONE_D_START, discrete_end_1d, sigma=SIGMA, n_paths=MIXTURE_PATHS, rng=rngs[5])
gmm_end_1d, gmm_idx_1d = _sample_gmm_endpoints(GMM_MEANS_1D, GMM_STDS_1D, GMM_WEIGHTS, n_samples=MIXTURE_PATHS, rng=rngs[6])
gmm_mix_1d = _sample_brownian_bridge_paths(time_grid, ONE_D_START, gmm_end_1d, sigma=SIGMA, n_paths=MIXTURE_PATHS, rng=rngs[7])

discrete_end_2d, discrete_idx_2d = _sample_discrete_endpoints(DISCRETE_ENDPOINTS_2D, DISCRETE_WEIGHTS, n_samples=MIXTURE_PATHS, rng=rngs[8])
discrete_mix_2d = _sample_brownian_bridge_paths(time_grid, TWO_D_START, discrete_end_2d, sigma=SIGMA, n_paths=MIXTURE_PATHS, rng=rngs[9])
gmm_end_2d, gmm_idx_2d = _sample_gmm_endpoints(GMM_MEANS_2D, GMM_STDS_2D, GMM_WEIGHTS, n_samples=MIXTURE_PATHS, rng=rngs[10])
gmm_mix_2d = _sample_brownian_bridge_paths(time_grid, TWO_D_START, gmm_end_2d, sigma=SIGMA, n_paths=MIXTURE_PATHS, rng=rngs[11])

validation_bridge_paths = _sample_brownian_bridge_paths(
    time_grid,
    ONE_D_START,
    ONE_D_END,
    sigma=SIGMA,
    n_paths=VALIDATION_SAMPLES,
    rng=rngs[12],
)
validation_key = jax.random.PRNGKey(RNG_SEED + 12345)
validation_keys = jax.random.split(validation_key, VALIDATION_SAMPLES)
validation_mid_jax = np.asarray(
    jax.vmap(
        lambda key: sample_brownian_bridge_state(
            jnp.asarray([ONE_D_START], dtype=jnp.float32),
            jnp.asarray([ONE_D_END], dtype=jnp.float32),
            t_mid,
            T_START,
            T_END,
            SIGMA,
            key,
        )
    )(validation_keys)
)[:, 0]

score_time_index_1d = int(round(0.58 * (len(time_grid) - 1)))
score_time_1d = float(time_grid[score_time_index_1d])
score_component_1d = int(np.argmax(DISCRETE_WEIGHTS))
score_path_idx_1d = int(np.flatnonzero(discrete_idx_1d == score_component_1d)[0])
score_path_1d = discrete_mix_1d[score_path_idx_1d]
score_endpoint_1d = float(discrete_end_1d[score_path_idx_1d, 0])
score_state_1d = float(score_path_1d[score_time_index_1d, 0])
score_value_1d = (score_endpoint_1d - score_state_1d) / max(T_END - score_time_1d, 1e-12)

posterior_time_index_1d = mid_index
posterior_time_1d = float(time_grid[posterior_time_index_1d])
posterior_component_1d = int(np.argmax(GMM_WEIGHTS))
posterior_path_idx_1d = int(np.flatnonzero(gmm_idx_1d == posterior_component_1d)[0])
posterior_state_1d = float(gmm_mix_1d[posterior_path_idx_1d, posterior_time_index_1d, 0])
posterior_indices_1d = _component_representative_indices(gmm_idx_1d, per_component=4)
posterior_endpoints_1d = gmm_end_1d[posterior_indices_1d, 0]
posterior_components_1d = gmm_idx_1d[posterior_indices_1d]
posterior_log_weights_1d = _log_brownian_transition_density(
    posterior_endpoints_1d[:, None],
    np.asarray([posterior_state_1d]),
    delta_t=T_END - posterior_time_1d,
    sigma=SIGMA,
) - _log_brownian_transition_density(
    posterior_endpoints_1d[:, None],
    np.asarray([ONE_D_START]),
    delta_t=T_END - T_START,
    sigma=SIGMA,
)
posterior_weights_1d = _normalize_log_weights(posterior_log_weights_1d)
posterior_scores_1d = (posterior_endpoints_1d - posterior_state_1d) / max(T_END - posterior_time_1d, 1e-12)
posterior_drift_1d = float(np.sum(posterior_weights_1d * posterior_scores_1d))
posterior_endpoint_mean_1d = float(np.sum(posterior_weights_1d * posterior_endpoints_1d))

score_time_index = int(round(0.58 * (len(time_grid) - 1)))
score_time = float(time_grid[score_time_index])
score_component = int(np.argmax(DISCRETE_WEIGHTS))
score_path_idx = int(np.flatnonzero(discrete_idx_2d == score_component)[0])
score_path = discrete_mix_2d[score_path_idx]
score_endpoint = discrete_end_2d[score_path_idx]
score_state = score_path[score_time_index]
score_vector = (score_endpoint - score_state) / max(T_END - score_time, 1e-12)

posterior_time_index = mid_index
posterior_time = float(time_grid[posterior_time_index])
posterior_component = int(np.argmax(GMM_WEIGHTS))
posterior_path_idx = int(np.flatnonzero(gmm_idx_2d == posterior_component)[0])
posterior_state = gmm_mix_2d[posterior_path_idx, posterior_time_index]
posterior_indices = _component_representative_indices(gmm_idx_2d, per_component=4)
posterior_endpoints = gmm_end_2d[posterior_indices]
posterior_components = gmm_idx_2d[posterior_indices]
posterior_log_weights = _log_brownian_transition_density(
    posterior_endpoints,
    posterior_state,
    delta_t=T_END - posterior_time,
    sigma=SIGMA,
) - _log_brownian_transition_density(
    posterior_endpoints,
    TWO_D_START,
    delta_t=T_END - T_START,
    sigma=SIGMA,
)
posterior_weights = _normalize_log_weights(posterior_log_weights)
posterior_score_vectors = (posterior_endpoints - posterior_state[None, :]) / max(T_END - posterior_time, 1e-12)
posterior_drift_vector = np.sum(posterior_weights[:, None] * posterior_score_vectors, axis=0)
posterior_endpoint_mean = np.sum(posterior_weights[:, None] * posterior_endpoints, axis=0)

presentation_sequence = [
    "one_d_anchored_reference",
    "one_d_pinned_bridge",
    "one_d_local_law_continuous",
    "one_d_local_law_empirical",
    "one_d_training_score_target",
    "one_d_posterior_drift_target",
]

print(f"time_grid shape        : {time_grid.shape}")
print(f"anchored_1d shape      : {anchored_1d.shape}")
print(f"pinned_1d shape        : {pinned_1d.shape}")
print(f"discrete_mix_2d shape  : {discrete_mix_2d.shape}")
print(f"validation mid samples : {validation_mid_jax.shape[0]}")
print(f"score target 1d idx/time: {score_path_idx_1d}, {score_time_1d:.3f}")
print(f"drift target 1d idx/time: {posterior_path_idx_1d}, {posterior_time_1d:.3f}")
print(f"score target 2d idx/time: {score_path_idx}, {score_time:.3f}")
print(f"posterior idx/time     : {posterior_path_idx}, {posterior_time:.3f}")
print(f"posterior reps         : {posterior_indices.tolist()}")


time_grid shape        : (121,)
anchored_1d shape      : (320, 121, 1)
pinned_1d shape        : (240, 121, 1)
discrete_mix_2d shape  : (240, 121, 2)
validation mid samples : 4096
score target 1d idx/time: 5, 0.583
drift target 1d idx/time: 0, 0.500
score target 2d idx/time: 1, 0.583
posterior idx/time     : 4, 0.500
posterior reps         : [4, 6, 11, 12, 0, 2, 7, 9, 1, 3, 5, 8]



## Sanity Checks

The notebook checks the three mathematical requirements from the implementation plan:

- the anchored terminal cloud matches the theoretical Brownian endpoint law,
- pinned paths hit their fixed endpoints exactly on the plotting grid,
- midpoint statistics from the pinned-bridge path sampler match the exact bridge moments used by `sample_brownian_bridge_state`.


In [19]:

anchored_terminal_1d = anchored_1d[:, -1, 0]
anchored_theory_mean = float(ONE_D_START)
anchored_theory_var = float(SIGMA * SIGMA * (T_END - T_START))
anchored_emp_mean = float(np.mean(anchored_terminal_1d))
anchored_emp_var = float(np.var(anchored_terminal_1d))

assert np.allclose(pinned_1d[:, 0, 0], ONE_D_START)
assert np.allclose(pinned_1d[:, -1, 0], ONE_D_END)
assert np.allclose(pinned_2d[:, 0, :], TWO_D_START[None, :])
assert np.allclose(pinned_2d[:, -1, :], TWO_D_END[None, :])

mid_empirical = validation_bridge_paths[:, mid_index, 0]
mid_theory_mean, mid_theory_var = _bridge_midpoint_moments(
    ONE_D_START,
    ONE_D_END,
    t_mid,
    t_start=T_START,
    t_end=T_END,
    sigma=SIGMA,
)
mid_emp_mean = float(np.mean(mid_empirical))
mid_emp_var = float(np.var(mid_empirical))
mid_jax_mean = float(np.mean(validation_mid_jax))
mid_jax_var = float(np.var(validation_mid_jax))

terminal_mean_tol = 0.10
terminal_var_tol = 0.18
mid_mean_tol = 0.06
mid_var_tol = 0.08

assert abs(anchored_emp_mean - anchored_theory_mean) < terminal_mean_tol
assert abs(anchored_emp_var - anchored_theory_var) < terminal_var_tol
assert abs(mid_emp_mean - mid_theory_mean) < mid_mean_tol
assert abs(mid_emp_var - mid_theory_var) < mid_var_tol
assert abs(mid_jax_mean - mid_theory_mean) < mid_mean_tol
assert abs(mid_jax_var - mid_theory_var) < mid_var_tol

sanity_checks = {
    "anchored_terminal": {
        "empirical_mean": anchored_emp_mean,
        "theoretical_mean": anchored_theory_mean,
        "empirical_var": anchored_emp_var,
        "theoretical_var": anchored_theory_var,
    },
    "pinned_endpoints_exact": True,
    "bridge_midpoint": {
        "time": t_mid,
        "empirical_mean": mid_emp_mean,
        "theoretical_mean": mid_theory_mean,
        "jax_mean": mid_jax_mean,
        "empirical_var": mid_emp_var,
        "theoretical_var": mid_theory_var,
        "jax_var": mid_jax_var,
    },
}

print(json.dumps(sanity_checks, indent=2))


{
  "anchored_terminal": {
    "empirical_mean": -1.2613648897317566,
    "theoretical_mean": -1.25,
    "empirical_var": 0.7499549315285098,
    "theoretical_var": 0.81
  },
  "pinned_endpoints_exact": true,
  "bridge_midpoint": {
    "time": 0.5,
    "empirical_mean": 0.13461472609863318,
    "theoretical_mean": 0.125,
    "jax_mean": 0.1299000382423401,
    "empirical_var": 0.2040013658697958,
    "theoretical_var": 0.2025,
    "jax_var": 0.20369037985801697
  }
}


In [ ]:

def _render_one_d_anchored_reference(base_path: Path) -> None:
    fig, axes = plt.subplots(2, 1, figsize=LOCAL_LAW_ONE_D_FIGSIZE, gridspec_kw={"height_ratios": [1.35, 1.0]})
    ax_paths, ax_hist = axes
    for path in anchored_1d:
        ax_paths.plot(time_grid, path[:, 0], color=REFERENCE_COLOR, alpha=0.10, linewidth=0.9)
    ax_paths.scatter([T_START], [ONE_D_START], color="black", s=32, zorder=5)
    ax_paths.set_title("Anchored Reference: Brownian Motion From A Fixed Left Endpoint")
    ax_paths.set_xlabel("time")
    ax_paths.set_ylabel("state")

    x_support = np.linspace(
        anchored_terminal_1d.min() - 0.4,
        anchored_terminal_1d.max() + 0.4,
        400,
    )
    ax_hist.hist(anchored_terminal_1d, bins=32, density=True, color="#b0b0b0", alpha=0.85)
    ax_hist.plot(
        x_support,
        _normal_pdf_1d(x_support, anchored_theory_mean, np.sqrt(anchored_theory_var)),
        color="black",
        linewidth=2.0,
    )
    ax_hist.set_xlabel(f"terminal state at t={T_END:g}")
    ax_hist.set_ylabel("density")
    ax_hist.set_title(r"$q_i^{z_{i-1}}(dy)$")
    fig.tight_layout()
    _save_figure(fig, base_path)


def _render_one_d_pinned_bridge(base_path: Path) -> None:
    fig, axes = plt.subplots(2, 1, figsize=LOCAL_LAW_ONE_D_FIGSIZE, gridspec_kw={"height_ratios": [2.4, 1.2]})
    ax_paths, ax_hist = axes
    for path in pinned_1d:
        ax_paths.plot(time_grid, path[:, 0], color=BRIDGE_COLOR, alpha=0.10, linewidth=0.95)
    ax_paths.scatter([T_START, T_END], [ONE_D_START, ONE_D_END], color="black", s=32, zorder=5)
    ax_paths.set_title("Pinned Brownian Bridge Between Two Fixed Endpoints")
    ax_paths.set_xlabel("time")
    ax_paths.set_ylabel("state")

    x_support = np.linspace(mid_empirical.min() - 0.4, mid_empirical.max() + 0.4, 400)
    ax_hist.hist(mid_empirical, bins=32, density=True, color=BRIDGE_COLOR, alpha=0.30)
    ax_hist.plot(
        x_support,
        _normal_pdf_1d(x_support, mid_theory_mean, np.sqrt(mid_theory_var)),
        color="black",
        linewidth=2.0,
    )
    ax_hist.set_xlabel(f"bridge state at t={t_mid:.3f}")
    ax_hist.set_ylabel("density")
    ax_hist.set_title("Exact midpoint law")
    fig.tight_layout()
    _save_figure(fig, base_path)



def _render_one_d_mixture(
    paths: np.ndarray,
    endpoints: np.ndarray,
    component_idx: np.ndarray,
    *,
    base_path: Path,
    title: str,
    density_kind: str,
    collapse_to_single: bool = False,
    survivor_path_index: int | None = None,
) -> None:
    fig, axes = plt.subplots(
        2,
        1,
        figsize=LOCAL_LAW_ONE_D_FIGSIZE,
        gridspec_kw={"height_ratios": [1.35, 1.0]},
    )
    ax_paths, ax_law = axes
    endpoint_values = endpoints[:, 0]
    endpoint_min = float(endpoint_values.min())
    endpoint_max = float(endpoint_values.max())
    fade_color = "#9aa3ad"
    survivor_color = BRIDGE_COLOR

    def _path_color(value: float, comp: int):
        if density_kind == "discrete":
            return _component_color(int(comp))
        if endpoint_max <= endpoint_min + 1e-12:
            return BRIDGE_COLOR
        color_level = (float(value) - endpoint_min) / (endpoint_max - endpoint_min)
        return plt.get_cmap("coolwarm")(float(np.clip(color_level, 0.0, 1.0)))

    if collapse_to_single and density_kind == "discrete":
        survivor_idx = int(0 if survivor_path_index is None else survivor_path_index)
        survivor_component = int(component_idx[survivor_idx])
        survivor_indices = np.flatnonzero(component_idx == survivor_component)
        highlight_indices = [survivor_idx]
        highlight_indices.extend(int(idx) for idx in survivor_indices if int(idx) != survivor_idx)
        highlight_indices = highlight_indices[:5]
        survivor_index_set = {int(idx) for idx in survivor_indices.tolist()}
        for idx, path in enumerate(paths):
            if idx in survivor_index_set:
                ax_paths.plot(time_grid, path[:, 0], color=survivor_color, alpha=0.18, linewidth=0.95)
            else:
                ax_paths.plot(time_grid, path[:, 0], color=fade_color, alpha=0.08, linewidth=0.80)
        for idx in highlight_indices:
            path = paths[idx]
            ax_paths.plot(time_grid, path[:, 0], color=survivor_color, alpha=0.95, linewidth=1.30)
        survivor_endpoint = float(endpoint_values[survivor_idx])
        ax_paths.scatter([T_END], [survivor_endpoint], color=survivor_color, s=34, zorder=6)
    else:
        path_colors = [_path_color(value, int(comp)) for value, comp in zip(endpoint_values, component_idx, strict=True)]
        for path, color in zip(paths, path_colors, strict=True):
            ax_paths.plot(time_grid, path[:, 0], color=color, alpha=0.18, linewidth=0.9)
        path_semantics = "color = terminal atom $y_k$" if density_kind == "discrete" else "color = sampled terminal endpoint $y$"
        ax_paths.text(
            0.03,
            0.75,
            path_semantics,
            transform=ax_paths.transAxes,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
        )
    ax_paths.scatter([T_START], [ONE_D_START], color="black", s=28, zorder=5)
    ax_paths.set_title(title)
    ax_paths.set_xlabel("time")
    ax_paths.set_ylabel("state")
    ax_paths.text(
        0.03,
        0.85,
        r"fixed left anchor $z_{i-1}$",
        transform=ax_paths.transAxes,
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
    )

    x_support = np.linspace(endpoint_min - 0.4, endpoint_max + 0.4, 400)
    reference_density = _normal_pdf_1d(x_support, anchored_theory_mean, np.sqrt(anchored_theory_var))
    ax_law.plot(x_support, reference_density, color=REFERENCE_COLOR, linewidth=1.6, linestyle="--")
    ax_law.text(
        0.03,
        0.70,
        r"gray dashed: $q_i^{z_{i-1}}$",
        transform=ax_law.transAxes,
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
    )
    if density_kind == "discrete":
        if collapse_to_single:
            survivor_idx = int(0 if survivor_path_index is None else survivor_path_index)
            survivor_component = int(component_idx[survivor_idx])
            endpoint_value = float(DISCRETE_ENDPOINTS_1D[survivor_component])
            weight_value = 1.0
            max_height = max(weight_value, float(reference_density.max()))
            ax_law.vlines(endpoint_value, 0.0, weight_value, color=survivor_color, linewidth=2.5, alpha=0.98)
            ax_law.scatter(endpoint_value, weight_value, color=survivor_color, s=68, edgecolor="white", linewidth=0.5, zorder=4, alpha=0.98)
            ax_law.text(endpoint_value, weight_value + 0.035 * max_height, f"{weight_value:.2f}", ha="center", va="bottom", color=survivor_color)
        else:
            max_height = max(float(DISCRETE_WEIGHTS.max()), float(reference_density.max()))
            for weight, endpoint, color in zip(DISCRETE_WEIGHTS, DISCRETE_ENDPOINTS_1D, MIXTURE_COLORS, strict=True):
                endpoint_value = float(endpoint)
                weight_value = float(weight)
                ax_law.vlines(endpoint_value, 0.0, weight_value, color=color, linewidth=2.2, alpha=0.95)
                ax_law.scatter(endpoint_value, weight_value, color=color, s=68, edgecolor="white", linewidth=0.5, zorder=4, alpha=0.95)
                ax_law.text(endpoint_value, weight_value + 0.035 * max_height, f"{weight_value:.2f}", ha="center", va="bottom", color=color)
        ax_law.set_ylabel("mass / density")
        if not collapse_to_single:
            ax_law.text(
                0.03,
                0.85,
                r"colored atoms: $\widehat\gamma_i(\cdot\mid\zeta_{i-1})$",
                transform=ax_law.transAxes,
                bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
            )
        ax_law.set_ylim(0.0, 1.28 * max_height)
    else:
        path_colors = [_path_color(value, int(comp)) for value, comp in zip(endpoint_values, component_idx, strict=True)]
        gamma_density = _mixture_pdf_1d(x_support, GMM_MEANS_1D, GMM_STDS_1D, GMM_WEIGHTS)
        ax_law.plot(x_support, gamma_density, color="#111111", linewidth=1.8)
        ax_law.scatter(endpoint_values, np.zeros_like(endpoint_values), c=path_colors, s=8, alpha=0.35, zorder=3)
        ax_law.set_ylabel("density")
        ax_law.text(
            0.03,
            0.85,
            r"black: $\gamma_i(\cdot\mid\zeta_{i-1})$",
            transform=ax_law.transAxes,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
        )
    ax_law.set_title(r"Conditional terminal law $\gamma_i(\cdot\mid\zeta_{i-1})$")
    ax_law.set_xlabel(f"terminal state at t={T_END:g}")
    fig.tight_layout(pad=0.35, h_pad=0.55)
    _save_figure(fig, base_path)


def _render_one_d_training_score_target(base_path: Path) -> None:
    fig, axes = plt.subplots(2, 1, figsize=(8.4, 6.6), gridspec_kw={"height_ratios": [2.15, 1.0]})
    ax_paths, ax_score = axes
    comp_mask = discrete_idx_1d == score_component_1d
    for path in discrete_mix_1d[comp_mask][:26]:
        ax_paths.plot(time_grid, path[:, 0], color=_component_color(score_component_1d), alpha=0.10, linewidth=0.9)
    ax_paths.plot(time_grid, score_path_1d[:, 0], color=_component_color(score_component_1d), linewidth=2.1, alpha=0.95)
    ax_paths.axvline(score_time_1d, color="#666666", linestyle="--", linewidth=1.2)
    ax_paths.scatter([T_START, score_time_1d, T_END], [ONE_D_START, score_state_1d, score_endpoint_1d], color=["black", "black", _component_color(score_component_1d)], s=[28, 36, 52], zorder=5)
    ax_paths.annotate(
        "",
        xy=(T_END, score_endpoint_1d),
        xytext=(score_time_1d, score_state_1d),
        arrowprops=dict(arrowstyle="->", color="#111111", linewidth=2.0),
        zorder=6,
    )
    ax_paths.text(score_time_1d + 0.015, score_state_1d + 0.08, r"$z$", fontsize=10)
    ax_paths.text(T_END - 0.035, score_endpoint_1d + 0.08, r"$y$", fontsize=10, color=_component_color(score_component_1d))
    ax_paths.set_title("Empirical Training Target: Fixed-Endpoint Score")
    ax_paths.set_xlabel("time")
    ax_paths.set_ylabel("state")

    endpoint_support = np.linspace(min(DISCRETE_ENDPOINTS_1D.min(), score_state_1d) - 0.3, max(DISCRETE_ENDPOINTS_1D.max(), score_state_1d) + 0.3, 300)
    score_support = (endpoint_support - score_state_1d) / max(T_END - score_time_1d, 1e-12)
    ax_score.plot(endpoint_support, score_support, color="#222222", linewidth=1.8)
    for idx, (endpoint, weight) in enumerate(zip(DISCRETE_ENDPOINTS_1D, DISCRETE_WEIGHTS, strict=True)):
        endpoint_value = float(endpoint)
        score_value = (endpoint_value - score_state_1d) / max(T_END - score_time_1d, 1e-12)
        ax_score.scatter(endpoint_value, score_value, color=_component_color(idx), s=65 + 140 * float(weight), edgecolor="white", linewidth=0.6, zorder=4)
    ax_score.scatter(score_endpoint_1d, score_value_1d, color=_component_color(score_component_1d), s=180, edgecolor="black", linewidth=0.9, zorder=5)
    ax_score.set_xlabel(r"endpoint $y$")
    ax_score.set_ylabel(r"$\mathsf{s}_i(t,z,y)$")
    ax_score.set_title(r"Score field at fixed $(t,z)$")
    fig.tight_layout()
    _save_figure(fig, base_path)


def _render_one_d_posterior_drift_target(base_path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.4), gridspec_kw={"width_ratios": [1.0, 1.05]})
    ax_weights, ax_scores = axes

    endpoint_support = np.linspace(gmm_end_1d[:, 0].min() - 0.4, gmm_end_1d[:, 0].max() + 0.4, 400)
    ax_weights.plot(endpoint_support, _mixture_pdf_1d(endpoint_support, GMM_MEANS_1D, GMM_STDS_1D, GMM_WEIGHTS), color="#8c8c8c", linewidth=1.7, linestyle="--")
    for endpoint, comp, weight in zip(posterior_endpoints_1d, posterior_components_1d, posterior_weights_1d, strict=True):
        ax_weights.vlines(endpoint, 0.0, weight, color=_component_color(int(comp)), linewidth=2.0 + 8.0 * float(weight), alpha=0.85)
        ax_weights.scatter(endpoint, weight, color=_component_color(int(comp)), s=45 + 260.0 * float(weight), edgecolor="white", linewidth=0.6, zorder=4)
    ax_weights.axvline(posterior_endpoint_mean_1d, color="#111111", linewidth=1.4, linestyle="-")
    ax_weights.axvline(posterior_state_1d, color="#555555", linewidth=1.1, linestyle=":")
    ax_weights.set_title("Posterior endpoint weights at fixed $(t,z)$")
    ax_weights.set_xlabel(r"endpoint $y$")
    ax_weights.set_ylabel("posterior weight")

    score_support = (endpoint_support - posterior_state_1d) / max(T_END - posterior_time_1d, 1e-12)
    ax_scores.plot(endpoint_support, score_support, color="#222222", linewidth=1.8)
    for endpoint, comp, weight, score in zip(posterior_endpoints_1d, posterior_components_1d, posterior_weights_1d, posterior_scores_1d, strict=True):
        ax_scores.scatter(endpoint, score, color=_component_color(int(comp)), s=50 + 260.0 * float(weight), edgecolor="white", linewidth=0.6, zorder=4)
    ax_scores.axhline(posterior_drift_1d, color="#111111", linewidth=2.4)
    ax_scores.axvline(posterior_endpoint_mean_1d, color="#111111", linewidth=1.1, linestyle="--")
    ax_scores.set_title("Posterior average of endpoint scores gives the drift")
    ax_scores.set_xlabel(r"endpoint $y$")
    ax_scores.set_ylabel(r"score / drift value")
    ax_scores.text(0.03, 0.05, r"$v_i^\gamma(z,\zeta,t)=\mathrm{E}[\mathsf{s}_i(t,z,Z_{t_i})\mid H_{i-1},Z_t=z]$", transform=ax_scores.transAxes, fontsize=9.5, bbox=dict(facecolor="white", alpha=0.78, edgecolor="none", pad=2.5))

    fig.tight_layout()
    _save_figure(fig, base_path)


def _plot_endpoint_cloud(ax: plt.Axes, x0: np.ndarray) -> None:
    theta = np.linspace(0.0, 2.0 * np.pi, 240)
    radius = SIGMA * np.sqrt(T_END - T_START)
    circle_x = x0[0] + radius * np.cos(theta)
    circle_y = x0[1] + radius * np.sin(theta)
    ax.fill(circle_x, circle_y, color="#d9d9d9", alpha=0.18, zorder=0)
    ax.plot(circle_x, circle_y, color="#b3b3b3", linewidth=1.1, linestyle="--", zorder=0)


def _configure_latent_axis(ax: plt.Axes) -> None:
    ax.set_xlabel("latent coordinate 1")
    ax.set_ylabel("latent coordinate 2")
    ax.set_aspect("equal", adjustable="box")


def _render_two_d_anchored_reference(base_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6.0, 5.6))
    _plot_endpoint_cloud(ax, TWO_D_START)
    for path in anchored_2d[:140]:
        ax.plot(path[:, 0], path[:, 1], color=REFERENCE_COLOR, alpha=0.12, linewidth=0.9)
    ax.scatter(anchored_2d[:, -1, 0], anchored_2d[:, -1, 1], color="#cfcfcf", s=14, alpha=0.25)
    ax.scatter([TWO_D_START[0]], [TWO_D_START[1]], color="black", s=34, zorder=5)
    ax.set_title("Anchored Reference Fan In 2D")
    _configure_latent_axis(ax)
    fig.tight_layout()
    _save_figure(fig, base_path)


def _render_two_d_pinned_bridge(base_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6.0, 5.6))
    for path in pinned_2d[:180]:
        ax.plot(path[:, 0], path[:, 1], color=BRIDGE_COLOR, alpha=0.12, linewidth=0.95)
    ax.scatter([TWO_D_START[0], TWO_D_END[0]], [TWO_D_START[1], TWO_D_END[1]], color="black", s=34, zorder=5)
    ax.set_title("Pinned Bridge Family In 2D")
    _configure_latent_axis(ax)
    fig.tight_layout()
    _save_figure(fig, base_path)



def _render_two_d_mixture(
    paths: np.ndarray,
    endpoints: np.ndarray,
    component_idx: np.ndarray,
    *,
    base_path: Path,
    title: str,
    endpoint_style: str,
    collapse_to_single: bool = False,
    survivor_path_index: int | None = None,
) -> None:
    fig, axes = plt.subplots(
        2,
        1,
        figsize=LOCAL_LAW_TWO_D_FIGSIZE,
        gridspec_kw={"height_ratios": [1.2, 1.0]},
    )
    ax_paths, ax_terminal = axes
    fade_color = "#9aa3ad"
    survivor_color = BRIDGE_COLOR

    _plot_endpoint_cloud(ax_paths, TWO_D_START)
    for path in anchored_2d[:90]:
        ax_paths.plot(path[:, 0], path[:, 1], color=REFERENCE_COLOR, alpha=0.08, linewidth=0.85, zorder=1)
    if collapse_to_single and endpoint_style == "discrete":
        survivor_idx = int(0 if survivor_path_index is None else survivor_path_index)
        survivor_component = int(component_idx[survivor_idx])
        survivor_indices = np.flatnonzero(component_idx == survivor_component)
        highlight_indices = [survivor_idx]
        highlight_indices.extend(int(idx) for idx in survivor_indices if int(idx) != survivor_idx)
        highlight_indices = highlight_indices[:5]
        survivor_index_set = {int(idx) for idx in survivor_indices.tolist()}
        for idx, path in enumerate(paths[:72]):
            if idx in survivor_index_set:
                ax_paths.plot(path[:, 0], path[:, 1], color=survivor_color, alpha=0.18, linewidth=0.95, zorder=2)
            else:
                ax_paths.plot(path[:, 0], path[:, 1], color=fade_color, alpha=0.08, linewidth=0.80, zorder=2)
        for idx in highlight_indices:
            path = paths[idx]
            ax_paths.plot(path[:, 0], path[:, 1], color=survivor_color, alpha=0.95, linewidth=1.30, zorder=3)
        survivor_path = paths[survivor_idx]
        ax_paths.scatter([survivor_path[-1, 0]], [survivor_path[-1, 1]], color=survivor_color, s=34, zorder=6)
    else:
        for path in paths[:72]:
            ax_paths.plot(path[:, 0], path[:, 1], color=BRIDGE_COLOR, alpha=0.16, linewidth=0.95, zorder=2)
        ax_paths.text(
            0.03,
            0.83,
            r"sampled paths from $\mathrm{P}_i^\gamma(\cdot\mid\zeta_{i-1})$",
            transform=ax_paths.transAxes,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
        )
    ax_paths.scatter([TWO_D_START[0]], [TWO_D_START[1]], color="black", s=36, zorder=5)
    ax_paths.set_title(title)
    ax_paths.text(
        0.03,
        0.93,
        r"fixed left anchor $z_{i-1}$",
        transform=ax_paths.transAxes,
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
    )
    _configure_latent_axis(ax_paths)

    _plot_endpoint_cloud(ax_terminal, TWO_D_START)
    if endpoint_style == "discrete":
        sizes = 130.0 + 500.0 * DISCRETE_WEIGHTS
        survivor_component = None
        if collapse_to_single:
            survivor_idx = int(0 if survivor_path_index is None else survivor_path_index)
            survivor_component = int(component_idx[survivor_idx])
        for idx, endpoint, weight, size in zip(range(len(DISCRETE_WEIGHTS)), DISCRETE_ENDPOINTS_2D, DISCRETE_WEIGHTS, sizes, strict=True):
            if collapse_to_single:
                tone = survivor_color if idx == survivor_component else fade_color
                alpha = 0.98 if idx == survivor_component else 0.45
            else:
                tone = _component_color(idx)
                alpha = 0.95
            ax_terminal.scatter(endpoint[0], endpoint[1], color=tone, s=float(size), edgecolor="white", linewidth=0.6, zorder=4, alpha=alpha)
            ax_terminal.text(endpoint[0] + 0.05, endpoint[1] + 0.05, f"{float(weight):.2f}", color=tone)
        if not collapse_to_single:
            ax_terminal.text(
                0.03,
                0.85,
                r"colored atoms: $\widehat\gamma_i(\cdot\mid\zeta_{i-1})$",
                transform=ax_terminal.transAxes,
                bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
            )
    else:
        ax_terminal.scatter(endpoints[:, 0], endpoints[:, 1], color="#111111", s=16, alpha=0.18, zorder=3)
        ax_terminal.text(
            0.03,
            0.85,
            r"black cloud: $y\sim\gamma_i(\cdot\mid\zeta_{i-1})$",
            transform=ax_terminal.transAxes,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
        )
    ax_terminal.text(
        0.03,
        0.70,
        r"gray dashed: $q_i^{z_{i-1}}$",
        transform=ax_terminal.transAxes,
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.0),
    )
    ax_terminal.set_title(r"Conditional terminal law $\gamma_i(\cdot\mid\zeta_{i-1})$")
    _configure_latent_axis(ax_terminal)
    fig.tight_layout(pad=0.35, h_pad=0.6)
    _save_figure(fig, base_path)


def _render_two_d_training_score_target(base_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6.4, 5.8))
    _plot_endpoint_cloud(ax, TWO_D_START)
    comp_mask = discrete_idx_2d == score_component
    for path in discrete_mix_2d[comp_mask][:18]:
        ax.plot(path[:, 0], path[:, 1], color=_component_color(score_component), alpha=0.10, linewidth=0.9)
    ax.plot(score_path[:, 0], score_path[:, 1], color=_component_color(score_component), linewidth=2.1, alpha=0.95)
    ax.scatter([TWO_D_START[0]], [TWO_D_START[1]], color="black", s=36, zorder=5)
    ax.scatter([score_endpoint[0]], [score_endpoint[1]], color=_component_color(score_component), s=110, edgecolor="white", linewidth=0.8, zorder=6)
    ax.scatter([score_state[0]], [score_state[1]], color="black", s=44, zorder=7)
    ax.annotate(
        "",
        xy=score_endpoint,
        xytext=score_state,
        arrowprops=dict(arrowstyle="->", color="#111111", linewidth=2.0),
        zorder=6,
    )
    ax.text(
        score_state[0] + 0.1,
        score_state[1] - 0.22,
        r"$\mathsf{s}_i(t,z,y)=\frac{y-z}{t_i-t}$",
        fontsize=10,
        color="#111111",
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2.5),
    )
    ax.set_title("Empirical Training Target: Fixed-Endpoint Score")
    _configure_latent_axis(ax)
    fig.tight_layout()
    _save_figure(fig, base_path)


def _render_two_d_posterior_drift_target(base_path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(11.6, 5.2), gridspec_kw={"width_ratios": [1.0, 1.05]})
    ax_weights, ax_drift = axes

    _plot_endpoint_cloud(ax_weights, TWO_D_START)
    ax_weights.scatter(gmm_end_2d[:, 0], gmm_end_2d[:, 1], c=[_component_color(idx) for idx in gmm_idx_2d], s=10, alpha=0.08, zorder=1)
    sizes = 120.0 + 850.0 * posterior_weights
    ax_weights.scatter(
        posterior_endpoints[:, 0],
        posterior_endpoints[:, 1],
        s=sizes,
        c=[_component_color(idx) for idx in posterior_components],
        alpha=0.72,
        edgecolor="white",
        linewidth=0.6,
        zorder=3,
    )
    ax_weights.scatter([posterior_state[0]], [posterior_state[1]], color="black", s=40, zorder=5)
    ax_weights.scatter([posterior_endpoint_mean[0]], [posterior_endpoint_mean[1]], color="#111111", marker="x", s=70, zorder=5)
    ax_weights.set_title("Posterior Endpoint Weights At A Fixed Interior State")
    _configure_latent_axis(ax_weights)

    _plot_endpoint_cloud(ax_drift, TWO_D_START)
    ax_drift.scatter([posterior_state[0]], [posterior_state[1]], color="black", s=40, zorder=6)
    for endpoint, comp, weight, score in zip(
        posterior_endpoints,
        posterior_components,
        posterior_weights,
        posterior_score_vectors,
        strict=True,
    ):
        color = _component_color(int(comp))
        ax_drift.scatter(endpoint[0], endpoint[1], color=color, s=35 + 240.0 * weight, alpha=0.75, edgecolor="white", linewidth=0.4, zorder=4)
        ax_drift.annotate(
            "",
            xy=posterior_state + 0.42 * score,
            xytext=posterior_state,
            arrowprops=dict(arrowstyle="->", color=color, linewidth=0.8 + 4.5 * weight, alpha=0.18 + 0.85 * weight),
            zorder=3,
        )
    ax_drift.annotate(
        "",
        xy=posterior_state + 0.42 * posterior_drift_vector,
        xytext=posterior_state,
        arrowprops=dict(arrowstyle="->", color="#111111", linewidth=3.2),
        zorder=7,
    )
    ax_drift.text(
        posterior_state[0] + 0.08,
        posterior_state[1] - 0.26,
        r"$v_i^\gamma(z,\zeta,t)=\mathrm{E}[\mathsf{s}_i(t,z,Z_{t_i})\mid H_{i-1},Z_t=z]$",
        fontsize=9.5,
        color="#111111",
        bbox=dict(facecolor="white", alpha=0.78, edgecolor="none", pad=2.5),
    )
    ax_drift.set_title("Posterior Average Of Endpoint Scores Gives The Drift")
    _configure_latent_axis(ax_drift)
    fig.tight_layout()
    _save_figure(fig, base_path)


def _render_overview(base_path: Path) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.0), sharex=True)
    ax0, ax1, ax2 = axes

    for path in anchored_1d[:120]:
        ax0.plot(time_grid, path[:, 0], color=REFERENCE_COLOR, alpha=0.10, linewidth=0.9)
    ax0.scatter([T_START], [ONE_D_START], color="black", s=28, zorder=5)
    ax0.set_title("Anchored Reference")

    for path in pinned_1d[:140]:
        ax1.plot(time_grid, path[:, 0], color=BRIDGE_COLOR, alpha=0.11, linewidth=0.95)
    ax1.scatter([T_START, T_END], [ONE_D_START, ONE_D_END], color="black", s=28, zorder=5)
    ax1.set_title("Pinned Bridge")

    for comp in range(len(MIXTURE_COLORS)):
        mask = discrete_idx_1d == comp
        for path in discrete_mix_1d[mask][:20]:
            ax2.plot(time_grid, path[:, 0], color=_component_color(comp), alpha=0.18, linewidth=0.9)
    ax2.scatter([T_START], [ONE_D_START], color="black", s=28, zorder=5)
    ax2.set_title("Bridge Mixture")

    for ax in axes:
        ax.set_xlabel("time")
        ax.set_ylabel("state")
    fig.tight_layout()
    _save_figure(fig, base_path)


IndentationError: expected an indented block after 'for' statement on line 178 (2497044851.py, line 181)

## Export Figures

The notebook exports anchored-reference, pinned-bridge, and local-law figures. The empirical local-law panels now emphasize the practical collapse to one surviving Brownian bridge while keeping the other hypothetical pinned bridges and atoms visible only as faded context.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

_render_one_d_anchored_reference(EXPORT_DIR / "one_d_anchored_reference")
_render_one_d_pinned_bridge(EXPORT_DIR / "one_d_pinned_bridge")
_render_one_d_mixture(
    discrete_mix_1d,
    discrete_end_1d,
    discrete_idx_1d,
    base_path=EXPORT_DIR / "one_d_mixture_discrete",
    title="Toy local bridge mixture with empirical atomic endpoint law",
    density_kind="discrete",
)
_render_one_d_mixture(
    gmm_mix_1d,
    gmm_end_1d,
    gmm_idx_1d,
    base_path=EXPORT_DIR / "one_d_mixture_gmm",
    title="Toy local bridge mixture with continuous endpoint law",
    density_kind="gmm",
)
_render_one_d_mixture(
    gmm_mix_1d,
    gmm_end_1d,
    gmm_idx_1d,
    base_path=EXPORT_DIR / "one_d_local_law_continuous",
    title="Localized bridge mixture with fixed left anchor",
    density_kind="gmm",
)
_render_one_d_mixture(
    discrete_mix_1d,
    discrete_end_1d,
    discrete_idx_1d,
    base_path=EXPORT_DIR / "one_d_local_law_empirical",
    title="Localized bridge mixture with fixed left anchor",
    density_kind="discrete",
    collapse_to_single=True,
    survivor_path_index=score_path_idx_1d,
)
_render_one_d_training_score_target(EXPORT_DIR / "one_d_training_score_target")
_render_one_d_posterior_drift_target(EXPORT_DIR / "one_d_posterior_drift_target")
_render_two_d_anchored_reference(EXPORT_DIR / "two_d_anchored_reference")
_render_two_d_pinned_bridge(EXPORT_DIR / "two_d_pinned_bridge")
_render_two_d_mixture(
    discrete_mix_2d,
    discrete_end_2d,
    discrete_idx_2d,
    base_path=EXPORT_DIR / "two_d_mixture_discrete",
    title="2D toy local bridge mixture with empirical atomic law",
    endpoint_style="discrete",
)
_render_two_d_mixture(
    gmm_mix_2d,
    gmm_end_2d,
    gmm_idx_2d,
    base_path=EXPORT_DIR / "two_d_mixture_gmm",
    title="2D toy local bridge mixture with continuous endpoint law",
    endpoint_style="gmm",
)
_render_two_d_mixture(
    gmm_mix_2d,
    gmm_end_2d,
    gmm_idx_2d,
    base_path=EXPORT_DIR / "two_d_local_law_continuous",
    title="Localized bridge mixture with fixed left anchor",
    endpoint_style="gmm",
)
_render_two_d_mixture(
    discrete_mix_2d,
    discrete_end_2d,
    discrete_idx_2d,
    base_path=EXPORT_DIR / "two_d_local_law_empirical",
    title="Localized bridge mixture with fixed left anchor",
    endpoint_style="discrete",
    collapse_to_single=True,
    survivor_path_index=score_path_idx,
)
_render_two_d_training_score_target(EXPORT_DIR / "two_d_training_score_target")
_render_two_d_posterior_drift_target(EXPORT_DIR / "two_d_posterior_drift_target")
_render_overview(EXPORT_DIR / "overview_three_laws")

exported_bases = [
    "one_d_anchored_reference",
    "one_d_pinned_bridge",
    "one_d_mixture_discrete",
    "one_d_mixture_gmm",
    "one_d_local_law_continuous",
    "one_d_local_law_empirical",
    "one_d_training_score_target",
    "one_d_posterior_drift_target",
    "two_d_anchored_reference",
    "two_d_pinned_bridge",
    "two_d_mixture_discrete",
    "two_d_mixture_gmm",
    "two_d_local_law_continuous",
    "two_d_local_law_empirical",
    "two_d_training_score_target",
    "two_d_posterior_drift_target",
    "overview_three_laws",
]

manifest = {
    "export_dir": str(EXPORT_DIR),
    "seed": RNG_SEED,
    "sigma": SIGMA,
    "time_interval": [T_START, T_END],
    "n_time": N_TIME,
    "transparent_export": TRANSPARENT_EXPORT,
    "exported_bases": exported_bases,
    "presentation_sequence": presentation_sequence,
    "discrete_weights": DISCRETE_WEIGHTS.tolist(),
    "discrete_endpoints_1d": DISCRETE_ENDPOINTS_1D.tolist(),
    "discrete_endpoints_2d": DISCRETE_ENDPOINTS_2D.tolist(),
    "gmm_weights": GMM_WEIGHTS.tolist(),
    "gmm_means_1d": GMM_MEANS_1D.tolist(),
    "gmm_stds_1d": GMM_STDS_1D.tolist(),
    "gmm_means_2d": GMM_MEANS_2D.tolist(),
    "gmm_stds_2d": GMM_STDS_2D.tolist(),
    "score_target": {
        "time": score_time_1d,
        "path_index": score_path_idx_1d,
        "component": score_component_1d,
        "state": score_state_1d,
        "endpoint": score_endpoint_1d,
        "score_value": score_value_1d,
    },
    "drift_target": {
        "time": posterior_time_1d,
        "path_index": posterior_path_idx_1d,
        "component": posterior_component_1d,
        "state": posterior_state_1d,
        "posterior_indices": posterior_indices_1d.tolist(),
        "posterior_endpoints": posterior_endpoints_1d.tolist(),
        "posterior_weights": posterior_weights_1d.tolist(),
        "posterior_scores": posterior_scores_1d.tolist(),
        "posterior_endpoint_mean": posterior_endpoint_mean_1d,
        "drift_value": posterior_drift_1d,
    },
    "score_target_2d": {
        "time": score_time,
        "path_index": score_path_idx,
        "component": score_component,
        "state": score_state.tolist(),
        "endpoint": score_endpoint.tolist(),
        "score_vector": score_vector.tolist(),
    },
    "drift_target_2d": {
        "time": posterior_time,
        "path_index": posterior_path_idx,
        "component": posterior_component,
        "state": posterior_state.tolist(),
        "posterior_indices": posterior_indices.tolist(),
        "posterior_weights": posterior_weights.tolist(),
        "posterior_endpoint_mean": posterior_endpoint_mean.tolist(),
        "drift_vector": posterior_drift_vector.tolist(),
    },
    "sanity_checks": sanity_checks,
}
(EXPORT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")

print(json.dumps(manifest, indent=2))


{
  "export_dir": "/data1/jy384/research/MMSFM/notebooks/figures/brownian_bridge_reference_mixture_viz",
  "seed": 7,
  "sigma": 0.9,
  "time_interval": [
    0.0,
    1.0
  ],
  "n_time": 121,
  "transparent_export": true,
  "exported_bases": [
    "one_d_anchored_reference",
    "one_d_pinned_bridge",
    "one_d_mixture_discrete",
    "one_d_mixture_gmm",
    "one_d_local_law_continuous",
    "one_d_local_law_empirical",
    "one_d_training_score_target",
    "one_d_posterior_drift_target",
    "two_d_anchored_reference",
    "two_d_pinned_bridge",
    "two_d_mixture_discrete",
    "two_d_mixture_gmm",
    "two_d_local_law_continuous",
    "two_d_local_law_empirical",
    "two_d_training_score_target",
    "two_d_posterior_drift_target",
    "overview_three_laws"
  ],
  "presentation_sequence": [
    "one_d_anchored_reference",
    "one_d_pinned_bridge",
    "one_d_local_law_continuous",
    "one_d_local_law_empirical",
    "one_d_training_score_target",
    "one_d_posterior_drift_ta